<a href="https://colab.research.google.com/github/dikshasaloni12/SupplyChain_KPI_Analysis/blob/main/SupplyChain_KPI_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
files = files.upload()

Saving supply_chain_data.csv to supply_chain_data.csv


In [15]:
import pandas as pd
df = pd.read_csv("supply_chain_data.csv")
df.head()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


I began with comprehensive data audit, profiling column types, summary statistics, and missing values. This ensures a clear understanding of the dataset's integrity and highlights potential anomalies before deeper analysis.

In [5]:
# Check structure and data types
df.info()

# Summary statistics for numeric columns
df.describe()

# Missing values per column
df.isnull().sum()

# Unique values per column
df.nunique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product type             100 non-null    object 
 1   SKU                      100 non-null    object 
 2   Price                    100 non-null    float64
 3   Availability             100 non-null    int64  
 4   Number of products sold  100 non-null    int64  
 5   Revenue generated        100 non-null    float64
 6   Customer demographics    100 non-null    object 
 7   Stock levels             100 non-null    int64  
 8   Lead times               100 non-null    int64  
 9   Order quantities         100 non-null    int64  
 10  Shipping times           100 non-null    int64  
 11  Shipping carriers        100 non-null    object 
 12  Shipping costs           100 non-null    float64
 13  Supplier name            100 non-null    object 
 14  Location                 10

,0
Product type,3
SKU,100
Price,100
Availability,63
Number of products sold,96
Revenue generated,100
Customer demographics,4
Stock levels,65
Lead times,29
Order quantities,61


# **1. Total Lead Time**

In [19]:
# Total lead time = manufacturing + shipping
df["Total Lead Time"] = df["Manufacturing lead time"] + df["Shipping times"]

# Preview
df[["Supplier name","Total Lead Time"]].head()

,Supplier name,Total Lead Time
0,Supplier 3,33
1,Supplier 3,32
2,Supplier 1,29
3,Supplier 5,24
4,Supplier 1,11


Total Lead Time captures the complete duration from production to delivery. This KPI is the backbone of supply chain efficiency analysis.

# **2. Delay Flag & Delay %**

In [20]:
df["Delay Flag"] = (df["Lead time"] > 20).astype(int)
df.groupby("Supplier name")["Delay Flag"].mean().sort_values()

,Delay Flag
Supplier name,
Supplier 4,0.277778
Supplier 1,0.333333
Supplier 5,0.500000
Supplier 3,0.533333
Supplier 2,0.545455


**Delay** Flag is a binary indicator of late shipments. Aggregating it gives Delay %, a direct measure of supplier reliability.

# **3. Cost per Unit**

In [21]:
df["Cost per Unit"] = df["Shipping costs"]/df["Order quantities"]
df.groupby("Shipping carriers")["Cost per Unit"].mean().sort_values()

,Cost per Unit
Shipping carriers,
Carrier B,0.211573
Carrier C,0.298138
Carrier A,0.386975


**Cost per Unit** normalizes shipping costs against order quantities, enabling fair comparison across carriers.

# **4. Defect Impact**

In [22]:
df["Defect Impact"] = df["Defect rates"] * df["Manufacturing costs"]
df.groupby("Supplier name")["Defect Impact"].mean().sort_values()

,Defect Impact
Supplier name,
Supplier 2,94.703073
Supplier 1,96.528918
Supplier 3,101.906794
Supplier 5,112.159146
Supplier 4,138.520687


**Defect Impact** translates quality failures into monetary risk, spotlighting suppliers whose defects are most expensive.

# **5. Supplier Performance Index (SPI)**

In [25]:
df["SPI"] = (
    (df["Lead time"] * 0.4) +
    (df["Defect rates"] * 0.3) +
    (df["Shipping costs"] * 0.3)
)
df.groupby("Supplier name")["SPI"].mean().sort_values()

,SPI
Supplier name,
Supplier 1,8.105891
Supplier 4,8.517980
Supplier 5,9.758794
Supplier 2,9.848760
Supplier 3,10.229700


**SPI** is a composite KPI integrating lead time, defect rate, and shipping cost. It provides a holistic ranking of supplier performance.

# **6. Carrier & Route Efficiency**

In [27]:
# Carrier reliability = delay % per carrier
carrier_reliability = (
    df.groupby("Shipping carriers")["Delay Flag"]
    .mean()
    .sort_values()
)
print(carrier_reliability)

# Route risk = avg defect rate + avg cost per route
route_risk = (
    df.groupby("Routes")[["Defect rates","Shipping costs"]]
    .mean()
)
print(route_risk)

Shipping carriers
Carrier C    0.379310
Carrier B    0.395349
Carrier A    0.535714
Name: Delay Flag, dtype: float64
         Defect rates  Shipping costs
Routes                               
Route A      2.341795        5.379699
Route B      2.322165        5.551986
Route C      2.054925        5.903218


**Carrier Reliability** highlights which carriers deliver on time, while Route Risk identifies logistics pathways with high defect rates and costs.

# **7. Export for Power BI**

In [28]:
df.to_csv("supply_chain_ready.csv", index=False)

In [29]:
from google.colab import files

# Export enriched dataset
df.to_csv("supply_chain_ready.csv", index=False)

# Download for Power BI visualization
files.download("supply_chain_ready.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

After engineering strategic KPIs—including **Total Lead Time**, **Delay Flag**, **Cost per Unit**, **Defect Impact**, and the **Supplier Performance Index (SPI)**—the enriched dataset was exported and downloaded for integration into Power BI. This step completes the analytical pipeline by transitioning from technical computation in Python to executive‑level visualization, enabling stakeholders to interpret supply chain performance through clear, actionable insights.